# Assault DDQN - HU004 Preflight + short training

Notebook orchestrator for the reproducible Assault DDQN pipeline. It bootstraps the selected GitHub ref, validates the HU002 environment contract, runs the HU004 preflight gate, and only then executes a short timestep-based DDQN training run.

## 1. Bootstrap Local -> GitHub -> Colab

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/j-mauro-r/reinforcement_learning_reto_1.git"
COLAB_ROOT = Path("/content/reinforcement_learning_reto_1")
BOOTSTRAP_REF = os.environ.get("ASSAULT_BOOTSTRAP_REF", "main")
BOOTSTRAP_COMMIT = os.environ.get("ASSAULT_BOOTSTRAP_COMMIT") or None
INSTALL_DEPENDENCIES = os.environ.get("ASSAULT_INSTALL_DEPENDENCIES", "1") == "1"


def _running_in_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return False
    return True


def _git_output(args, cwd):
    return subprocess.check_output(["git", *args], cwd=str(cwd), text=True).strip()


if _running_in_colab():
    if not (COLAB_ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--prune", "origin"], cwd=str(COLAB_ROOT), check=True)
    provisional_ref = BOOTSTRAP_COMMIT or f"origin/{BOOTSTRAP_REF}"
    provisional_sha = _git_output(["rev-parse", "--verify", f"{provisional_ref}^{{commit}}"], COLAB_ROOT)
    subprocess.run(["git", "checkout", "--detach", provisional_sha], cwd=str(COLAB_ROOT), check=True)
    ASSAULT_DIR = COLAB_ROOT / "2_Assault"
else:
    PROJECT_ROOT = Path(_git_output(["rev-parse", "--show-toplevel"], Path.cwd()))
    ASSAULT_DIR = PROJECT_ROOT / "2_Assault"

for path in (ASSAULT_DIR, ASSAULT_DIR.parent):
    value = str(path.resolve())
    if value in sys.path:
        sys.path.remove(value)
    sys.path.insert(0, value)

from src.execution_bootstrap import (
    install_project_requirements,
    prepare_execution_environment,
    verify_environment_import,
)

bootstrap = prepare_execution_environment(
    requested_ref=BOOTSTRAP_REF,
    requested_commit=BOOTSTRAP_COMMIT,
    repo_url=REPO_URL,
    colab_root=COLAB_ROOT,
)

PROJECT_ROOT = bootstrap.repo_root
ASSAULT_DIR = bootstrap.assault_dir

if INSTALL_DEPENDENCIES:
    install_project_requirements(bootstrap.requirements_path)

environment_source = verify_environment_import(bootstrap)
bootstrap.as_dict()


## 2. Imports and configuration

In [ ]:
from src.agent import DDQNAgent
from src.environment import create_assault_env, get_environment_metadata, validate_frameskip_once
from src.preflight import run_preflight_checks
from src.replay_buffer import ReplayBuffer
from src.trainer import Trainer
from src.utils import get_runtime_info, load_yaml_config

config = load_yaml_config(ASSAULT_DIR / "configs" / "ddqn_config.yaml")
seed = int(config["reproducibility"]["seed"])
print("PROJECT_ROOT:", PROJECT_ROOT)
print("ASSAULT_DIR:", ASSAULT_DIR)
print("BOOTSTRAP_REF:", BOOTSTRAP_REF)
print("BOOTSTRAP_COMMIT:", BOOTSTRAP_COMMIT or "<none>")
print("EXECUTED_SHA:", bootstrap.resolved_sha)
print("src.environment:", environment_source)
config


## 3. Runtime and hardware

In [ ]:
runtime_info = get_runtime_info()
runtime_info


## 4. HU002 environment contract

In [ ]:
train_env = create_assault_env(config, mode="train", seed=seed)
eval_env = create_assault_env(config, mode="eval", seed=seed + 1)

obs, info = train_env.reset(seed=seed)
metadata = get_environment_metadata(train_env, config, mode="train", seed=seed)

print("Observation shape:", obs.shape)
print("Observation dtype:", obs.dtype)
print("Action space:", train_env.action_space)
print("Action meanings:", train_env.unwrapped.get_action_meanings())
print("Initial info:", info)
print("Metadata:", metadata)


## 5. HU002 autovalidations

In [ ]:
assert obs.shape == (4, 84, 84)
assert str(obs.dtype) == "uint8"
assert train_env.action_space.n == 7
assert train_env.observation_space.shape == eval_env.observation_space.shape
assert train_env.observation_space.dtype == eval_env.observation_space.dtype
assert validate_frameskip_once(train_env, expected_frameskip=4, steps=5)

obs, info = train_env.reset(seed=seed)
for step in range(100):
    action = int(train_env.action_space.sample())
    obs, reward, terminated, truncated, info = train_env.step(action)
    assert obs.shape == (4, 84, 84)
    assert str(obs.dtype) == "uint8"
    if terminated or truncated:
        obs, info = train_env.reset()

print("HU002 validations passed.")
train_env.close()
eval_env.close()


## 6. HU004 preflight gate

In [ ]:
preflight_report = run_preflight_checks(config)
print(preflight_report.format_summary())
preflight_report.as_dict()


## 7. Abort if preflight fails

In [ ]:
if not preflight_report.ready_for_training:
    raise RuntimeError("READY_FOR_TRAINING=False; HU004 short training aborted.")
print("READY_FOR_TRAINING=True")


## 8. HU004 short timestep training

In [ ]:
training_env = create_assault_env(config, mode="train", seed=seed)
try:
    device = preflight_report.device
    agent = DDQNAgent(config, device=device, seed=seed)
    replay_buffer = ReplayBuffer(
        capacity=int(config["replay_buffer"]["capacity"]),
        seed=seed,
    )
    trainer = Trainer(training_env, agent, replay_buffer, config)
    training_summary = trainer.train()
finally:
    training_env.close()

training_summary.as_dict()


## 9. HU004 result

In [ ]:
assert training_summary.global_step == int(config["training"]["total_timesteps"])
assert training_summary.transitions_stored == training_summary.global_step
assert training_summary.updates_count > 0
assert training_summary.last_loss is not None
assert training_summary.online_weights_changed

print("HU004 short training PASS")
print("global_step:", training_summary.global_step)
print("transitions_stored:", training_summary.transitions_stored)
print("updates_count:", training_summary.updates_count)
print("last_loss:", training_summary.last_loss)
print("mean_loss:", training_summary.mean_loss)
print("epsilon_initial:", training_summary.epsilon_initial)
print("epsilon_final:", training_summary.epsilon_final)
print("target_sync_steps:", training_summary.target_sync_steps)
print("episodes_completed:", training_summary.episodes_completed)
print("episode_rewards:", training_summary.episode_rewards)
print("episode_lengths:", training_summary.episode_lengths)
